# 05. Output Tuning and Validation\n\nThis is the fifth and final production stage. It does not clean data or estimate models. It reads the Stata and R outputs from intermediate, constructs publication-ready figures and LaTeX tables, places all final artifacts directly in figuresNtables, and validates both TeX documents.\n\nRun this notebook once after stages 01 through 04.


## 1. Setup and publication conventions\n\nAll figures use the common blue, sky-blue, and grey palette. Titles and panel labels are supplied by LaTeX rather than embedded in image files.


In [1]:

from pathlib import Path
import math
import shutil
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "data").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "data").is_dir():
    raise FileNotFoundError("Run this notebook from the replication-package folder.")

CODE_DIR = PROJECT_ROOT
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = RAW_DIR
INPUT_DIR = PROJECT_ROOT / "data" / "prepared"
INTERMEDIATE_DIR = PROJECT_ROOT / "intermediate"
FINAL_DIR = PROJECT_ROOT / "figuresNtables"
LOGS_DIR = PROJECT_ROOT / "logs"

TABLES_DIR = INTERMEDIATE_DIR
PAPER_FIGURES_DIR = FINAL_DIR
FIGURES_DIR = FINAL_DIR
OUTPUT_ROOT = FINAL_DIR
UPLOAD_DIR = FINAL_DIR

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
for directory in [INPUT_DIR, INTERMEDIATE_DIR, FINAL_DIR, LOGS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

HIGH_CUTOFF = 0.1169
EVENT_PERIOD = pd.Timestamp("2022-11-01")
EVENT_MIN, EVENT_MAX = -21, 40

NAVY = "#08519C"
SKY = "#56B4E9"
GREY = "#7F8C8D"
LIGHT_GREY = "#D9E2E8"

plt.rcParams.update({
    "figure.dpi": 140, "savefig.dpi": 320, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.22,
    "grid.color": "#9AA4AD", "axes.axisbelow": True,
})

def stars(beta, se):
    if pd.isna(beta) or pd.isna(se) or se <= 0:
        return ""
    p = math.erfc(abs(beta / se) / math.sqrt(2))
    return "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.10 else ""

def coefficient(beta, se, digits=3):
    suffix = stars(beta, se)
    value = f"{beta:.{digits}f}"
    return value + (rf"$^{{{suffix}}}$" if suffix else "")

def standard_error(se, digits=3):
    return f"({se:.{digits}f})" if pd.notna(se) else ""

def latex_escape(value):
    text = str(value)
    replacements = {"\\": r"\textbackslash{}", "&": r"\&", "%": r"\%",
                    "$": r"\$", "#": r"\#", "_": r"\_", "{": r"\{", "}": r"\}"}
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text

def read_first(filename):
    path = INTERMEDIATE_DIR / filename
    if not path.exists():
        return None
    frame = pd.read_csv(path)
    return frame.iloc[0] if len(frame) else None

def avg_row(specification, outcome):
    return read_first(f"twfe_average_{specification}_{outcome}.csv")

def longdiff_row(specification, outcome):
    return read_first(f"twfe_longdiff_{specification}_{outcome}.csv")

def require(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Required estimator output is missing: {path}")
    return path

def save_tex(filename, lines):
    path = INTERMEDIATE_DIR / filename
    path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    print(f"Built {path.name}")
    return path


## 4. Descriptive Patterns

Construct the two descriptive panels directly from the standardized total-CNO4 panel. Each CNO4 outcome is indexed to November 2022 = 100 before taking equal-weighted means within exposure groups. The groups are zero, middle (positive and below 0.1169), and high (at or above the 75th percentile, 0.1169).


In [2]:
total_panel = pd.read_csv(INPUT_DIR / "est_total_cno4.csv", dtype={"cno4": str})
total_panel["period_date"] = pd.to_datetime(total_panel["period"] + "-01")
total_panel["exposure_group"] = np.select(
    [
        total_panel["exposure_nearest"].eq(0),
        total_panel["exposure_nearest"].ge(HIGH_CUTOFF),
    ],
    ["Zero exposure", "High exposure"],
    default="Middle exposure",
)
reference = (
    total_panel.loc[
        total_panel["period_date"].eq(EVENT_PERIOD),
        ["cno4", "parados", "contratos"],
    ]
    .rename(columns={"parados": "parados_reference", "contratos": "contratos_reference"})
)
total_panel = total_panel.merge(reference, on="cno4", how="left", validate="many_to_one")
for outcome in ["parados", "contratos"]:
    denominator = total_panel[f"{outcome}_reference"]
    total_panel[f"{outcome}_index"] = np.where(
        denominator.gt(0),
        100 * total_panel[outcome] / denominator,
        np.nan,
    )

monthly = (
    total_panel.groupby(["period_date", "exposure_group"], as_index=False)
    .agg(
        parados_index=("parados_index", "mean"),
        contratos_index=("contratos_index", "mean"),
        n_parados=("parados_index", "count"),
        n_contratos=("contratos_index", "count"),
    )
)

styles = {
    "Zero exposure": (GREY, "--"),
    "Middle exposure": (SKY, "-"),
    "High exposure": (NAVY, "-"),
}
for outcome, ylabel, ylim, filename in [
    ("parados_index", "Index, November 2022 = 100", (70, 145), "Figure1_panelA.png"),
    ("contratos_index", "Index, November 2022 = 100", (40, 180), "Figure1_panelB.png"),
]:
    fig, ax = plt.subplots(figsize=(6.2, 3.7))
    for label in ["Zero exposure", "Middle exposure", "High exposure"]:
        subset = monthly.loc[monthly["exposure_group"].eq(label)]
        color, linestyle = styles[label]
        ax.plot(
            subset["period_date"],
            subset[outcome],
            label=label,
            color=color,
            linestyle=linestyle,
            linewidth=1.45,
        )
    ax.axvline(EVENT_PERIOD, color=GREY, linestyle=":", linewidth=1)
    ax.set_xlabel("Month")
    ax.set_ylabel(ylabel)
    ax.set_ylim(*ylim)
    ax.legend(frameon=False, fontsize=8, loc="upper right")
    fig.tight_layout()
    fig.savefig(PAPER_FIGURES_DIR / filename, bbox_inches="tight")
    plt.close(fig)

monthly.to_csv(TABLES_DIR / "descriptive_patterns_by_exposure_v1.csv", index=False)
print("Saved Figure 1 panels.")


Saved Figure 1 panels.


## 5. Preferred TWFE Event Study

Render the preferred CNO1-by-month specification from the Stata coefficient files. Both panels use the complete aligned event window from -21 through 40, with October 2022 normalized to zero.


In [3]:
def render_event_file(source, destination, ylim, ylabel="Estimated marginal effect", ytick_step=None):
    frame = pd.read_csv(require(source)).sort_values("event_time")
    event_grid = pd.DataFrame({"event_time": np.arange(EVENT_MIN, EVENT_MAX + 1)})
    frame = event_grid.merge(frame, on="event_time", how="left")
    fig, ax = plt.subplots(figsize=(6.2, 3.8))
    ax.axhline(0, color=GREY, linewidth=0.7)
    ax.axvline(0, color=GREY, linestyle="--", linewidth=0.8)
    ax.fill_between(
        frame["event_time"].astype(float),
        frame["ci_low"].astype(float),
        frame["ci_high"].astype(float),
        color=SKY,
        alpha=0.25,
        linewidth=0,
    )
    ax.plot(frame["event_time"], frame["estimate"], color=NAVY, linewidth=1.25)
    ax.scatter(frame["event_time"], frame["estimate"], color=NAVY, s=9, zorder=3)
    ax.set_xlim(EVENT_MIN, EVENT_MAX)
    ax.set_ylim(*ylim)
    if ytick_step is not None:
        ax.set_yticks(np.arange(ylim[0], ylim[1] + ytick_step / 2, ytick_step))
    ax.set_xticks(np.arange(-20, 41, 10))
    ax.set_xlabel("Months relative to November 2022")
    ax.set_ylabel(ylabel)
    fig.tight_layout()
    fig.savefig(destination, bbox_inches="tight")
    plt.close(fig)

render_event_file(
    TABLES_DIR / "twfe_event_preferred_cno1_month_ln_parados.csv",
    PAPER_FIGURES_DIR / "Figure2_panelA.png",
    (-0.05, 0.05),
    ytick_step=0.025,
)
render_event_file(
    TABLES_DIR / "twfe_event_preferred_cno1_month_ln_contratos.csv",
    PAPER_FIGURES_DIR / "Figure2_panelB.png",
    (-0.20, 0.20),
    ytick_step=0.05,
)
print("Saved preferred TWFE event-study panels.")


Saved preferred TWFE event-study panels.


## 6. Main Impact Table and Pre-Trend Diagnostics

Table 2 compares the unconditional TWFE benchmark with the preferred within-CNO1 specification using a direct November 2022-November 2025 long difference. The first-difference regression contains one observation per occupation, so the total-CNO4 sample can never exceed 502 observations. The two preferred-specification columns differ only in whether inference permits correlation within CNO4 or within broader CNO3 groups.

The appendix diagnostic table reports joint-nullity and coefficient-equality tests over the full, early, and recent pre-treatment windows, comparing inference clustered by CNO4 and CNO3.


In [4]:
labels = {
    "ln_parados": "Registered unemployed",
    "ln_contratos": "New contracts",
    "joint_equal_zero": "Jointly zero",
    "joint_equal_coefficients": "Equal to one another",
}
window_labels = {
    "full_-21_-2": "Full: -21 to -2",
    "early_-21_-10": "Early: -21 to -10",
    "recent_-10_-2": "Recent: -10 to -2",
}
cluster_sources = {
    "CNO4": [
        TABLES_DIR / "twfe_pretrend_preferred_cno1_month_ln_parados.csv",
        TABLES_DIR / "twfe_pretrend_preferred_cno1_month_ln_contratos.csv",
    ],
    "CNO3": [
        TABLES_DIR / "twfe_pretrend_preferred_cno1_month_cluster_cno3_ln_parados.csv",
        TABLES_DIR / "twfe_pretrend_preferred_cno1_month_cluster_cno3_ln_contratos.csv",
    ],
}
pretrend_frames = []
for cluster_level, paths in cluster_sources.items():
    for path in paths:
        frame = pd.read_csv(require(path))
        frame["cluster_level"] = cluster_level
        pretrend_frames.append(frame)
pretrend = pd.concat(pretrend_frames, ignore_index=True)
pretrend.to_csv(TABLES_DIR / "pretrend_diagnostics_v1.csv", index=False)

def pretrend_pvalue(cluster_level, outcome, window, test):
    match = pretrend.loc[
        pretrend["cluster_level"].eq(cluster_level)
        & pretrend["outcome"].eq(outcome)
        & pretrend["window"].eq(window)
        & pretrend["test"].eq(test),
        "p_value",
    ]
    if len(match) != 1:
        raise ValueError(f"Expected one pre-trend result, found {len(match)}")
    return float(match.iloc[0])

pre_lines = [
    r"\begin{table}[!htbp]",
    r"\centering",
    r"\caption{Pre-treatment coefficient tests under alternative clustering levels}",
    r"\label{tab:v1_pretrends}",
    r"\begin{threeparttable}",
    r"\scriptsize",
    r"\setlength{\tabcolsep}{4pt}",
    r"\begin{tabular}{llcccc}",
    r"\toprule",
    r"& & \multicolumn{2}{c}{CNO4 clustering} & \multicolumn{2}{c}{CNO3 clustering} \\",
    r"\cmidrule(lr){3-4}\cmidrule(lr){5-6}",
    r"Outcome & Window & Joint nullity & Equality & Joint nullity & Equality \\",
    r"\midrule",
]
for outcome_number, outcome in enumerate(["ln_parados", "ln_contratos"]):
    if outcome_number:
        pre_lines.append(r"\addlinespace")
    for window in ["full_-21_-2", "early_-21_-10", "recent_-10_-2"]:
        values = [
            pretrend_pvalue(cluster, outcome, window, test)
            for cluster in ["CNO4", "CNO3"]
            for test in ["joint_equal_zero", "joint_equal_coefficients"]
        ]
        pre_lines.append(
            f"{labels[outcome]} & {window_labels[window]} & "
            + " & ".join(f"{value:.3f}" for value in values)
            + r" \\"
        )
pre_lines += [
    r"\bottomrule",
    r"\end{tabular}",
    r"\begin{tablenotes}[flushleft]",
    r"\footnotesize",
    r"\item \emph{Notes:} Entries are $p$-values from Wald tests of the pre-treatment coefficients in the preferred continuous-treatment specification, which includes CNO4 and CNO1-by-year-month fixed effects. The joint-nullity test evaluates whether all coefficients in the indicated window equal zero; the equality test evaluates whether they equal one another while allowing their common value to differ from zero. The full, early, and recent windows cover event times -21 to -2, -21 to -10, and -10 to -2, respectively. October 2022 (event time -1) is the omitted reference month. The first two columns allow arbitrary correlation within 502 CNO4 occupations, while the final two allow correlation within 170 CNO3 occupation groups.",
    r"\end{tablenotes}",
    r"\end{threeparttable}",
    r"\end{table}",
]
save_tex("pretrend_diagnostics_v1.tex", pre_lines)


Built pretrend_diagnostics_v1.tex


WindowsPath('C:/Users/ngonzalezp/OneDrive - AIREF/Escritorio/Unempolyment_Benefits/ai_unemployment_analysis/code/last_scripts/intermediate/pretrend_diagnostics_v1.tex')

## 7. Identifying Support and TWFE Robustness

The preferred specification identifies exposure effects from CNO4 occupations with different exposure scores inside the same CNO1 family and month. The support figure and table quantify this within-family variation before presenting increasingly demanding CNO2-by-month controls.

The consolidated robustness table changes one design choice at a time: outcome transformation, exposure measure, geographic disaggregation, and occupation-family time controls.


In [ ]:
support_cno1 = pd.read_csv(require(TABLES_DIR / "exposure_support_within_cno1d.csv"))
support_cno2 = pd.read_csv(require(TABLES_DIR / "exposure_support_within_cno2.csv"))

for frame, label, filename in [
    (support_cno1, "Within-CNO1 standard deviation", "Support_panelA.png"),
    (support_cno2, "Within-CNO2 standard deviation", "Support_panelB.png"),
]:
    values = frame["sd_exposure"].fillna(0)
    fig, ax = plt.subplots(figsize=(5.7, 3.5))
    ax.hist(values, bins=min(15, max(5, len(values))), color=SKY, edgecolor="white")
    ax.axvline(values.median(), color=NAVY, linewidth=1.3, linestyle="--")
    ax.set_xlabel(label)
    ax.set_ylabel("Number of occupation families")
    fig.tight_layout()
    fig.savefig(PAPER_FIGURES_DIR / filename, bbox_inches="tight")
    plt.close(fig)

support_summary = pd.DataFrame([
    {
        "level": "CNO1",
        "families": len(support_cno1),
        "median_sd": support_cno1["sd_exposure"].fillna(0).median(),
        "median_range": support_cno1["range_exposure"].median(),
        "zero_range_share": (support_cno1["range_exposure"] == 0).mean(),
    },
    {
        "level": "CNO2",
        "families": len(support_cno2),
        "median_sd": support_cno2["sd_exposure"].fillna(0).median(),
        "median_range": support_cno2["range_exposure"].median(),
        "zero_range_share": (support_cno2["range_exposure"] == 0).mean(),
    },
])
support_summary.to_csv(TABLES_DIR / "family_support_summary_v1.csv", index=False)

support_tex = [
    r"\begin{table}[!htbp]",
    r"\centering",
    r"\caption{Within-family variation in AI exposure}",
    r"\label{tab:v1_family_support}",
    r"\begin{threeparttable}",
    r"\begin{tabular}{lcccc}",
    r"\toprule",
    r"Family level & Families & Median SD & Median range & Zero-range share \\",
    r"\midrule",
]
for row in support_summary.itertuples():
    support_tex.append(
        f"{row.level} & {row.families} & {row.median_sd:.3f} & "
        f"{row.median_range:.3f} & {100*row.zero_range_share:.1f}\\% \\\\"
    )
support_tex += [
    r"\bottomrule",
    r"\end{tabular}",
    r"\begin{tablenotes}[flushleft]",
    r"\footnotesize",
    r"\item \emph{Notes:} Statistics are computed across the 502 CNO4 occupations using the nearest-neighbor exposure score. A zero range means that all CNO4 occupations inside the family have the same exposure. Smaller within-family dispersion implies weaker identifying variation after family-by-month effects are absorbed.",
    r"\end{tablenotes}",
    r"\end{threeparttable}",
    r"\end{table}",
]
save_tex("family_support_v1.tex", support_tex)

robustness_panels = [
    ("Panel A. Baseline specification", ["benchmark_twfe", "preferred_cno1_month", "preferred_cno1_month_cluster_cno3"], "ln_parados", "ln_contratos"),
    ("Panel B. Alternative outcomes: log(Y+1)", ["benchmark_log_plus_one", "log_plus_one_cno1_month", "log_plus_one_cno1_month_cluster_cno3"], "ln_parados_p1", "ln_contratos_p1"),
    ("Panel C. Alternative exposure: cosine-weighted", ["benchmark_cosine_weighted", "cosine_weighted_cno1_month", "cosine_weighted_cno1_month_cluster_cno3"], "ln_parados", "ln_contratos"),
]
robust_tex = [
    r"\begin{table}[H]",
    r"\centering",
    r"\caption{Robustness checks: alternative outcomes and exposure measures}",
    r"\label{tab:v1_robustness}",
    r"\begin{threeparttable}",
    r"\scriptsize",
    r"\setlength{\tabcolsep}{3.5pt}",
    r"\begin{tabular}{lcccccc}",
    r"\toprule",
    r"& \multicolumn{3}{c}{\# of registered unemployed} & \multicolumn{3}{c}{\# of new contracts} \\",
    r"\cmidrule(lr){2-4}\cmidrule(lr){5-7}",
    r"& (1) & (2) & (3) & (4) & (5) & (6) \\",
    r"\midrule",
]
for panel_number, (label, specs, outcome_u, outcome_c) in enumerate(robustness_panels):
    if panel_number:
        robust_tex.append(r"\addlinespace")
    rows = [longdiff_row(spec, outcome_u) for spec in specs] + [longdiff_row(spec, outcome_c) for spec in specs]
    if any(row is None for row in rows):
        raise FileNotFoundError(f"Missing long-difference outputs for {label}")
    robust_tex.append(rf"\multicolumn{{7}}{{l}}{{\textit{{{label}}}}} \\")
    robust_tex.append("AI exposure & " + " & ".join(coefficient(row.estimate, row.se) for row in rows) + r" \\")
    robust_tex.append(" & " + " & ".join(standard_error(row.se) for row in rows) + r" \\")
    robust_tex.append("Impact of a 10 pp increase (percent) & " + " & ".join(f"{100*row.estimate:.1f}" for row in rows) + r" \\")
    robust_tex.append("Observations & " + " & ".join(f"{int(row.observations):,}" for row in rows) + r" \\")
robust_tex += [
    r"\midrule",
    r"CNO1 fixed effects & No & Yes & Yes & No & Yes & Yes \\",
    r"Clustered standard errors & CNO4 & CNO4 & CNO3 & CNO4 & CNO4 & CNO3 \\",
    r"\bottomrule",
    r"\end{tabular}",
    r"\begin{tablenotes}[flushleft]",
    r"\footnotesize",
    r"\item \emph{Notes:} Entries are occupation-level long-difference estimates between November 2022 and November 2025. Panel A reproduces the baseline. Panel B replaces each log outcome with log(Y+1). Panel C replaces the nearest-neighbor score with the cosine-weighted measure. Columns 1 and 4 are unconditional first-difference regressions; columns 2, 3, 5, and 6 absorb CNO1 fixed effects. Standard errors are clustered as indicated. Every exposure measure is divided by 0.10, so coefficients correspond to a 10 percentage-point increase in the indicated score. The impact rows report 100 times the coefficient and carry no significance symbols. $^{***}p<0.01$, $^{**}p<0.05$, and $^{*}p<0.10$.",
    r"\end{tablenotes}",
    r"\end{threeparttable}",
    r"\end{table}",
]
save_tex("robustness_checks_v1.tex", robust_tex)

# Occupation-level rank correlations across paper exposure measures.
from lib.jev_robustness import build_exposure_correlation_matrix, build_jev_robustness_outputs

jev_robustness_paths = build_jev_robustness_outputs(
    estimates_dir=TABLES_DIR,
    output_dir=PAPER_FIGURES_DIR,
)
exposure_matrix_status = build_exposure_correlation_matrix(
    project_root=PROJECT_ROOT,
    prepared_panel=INPUT_DIR / "est_total_cno4_jev.csv",
    jev_estimates=RAW_DIR / "jev_occupation_estimates.csv",
    output_dir=PAPER_FIGURES_DIR,
)
print("Jev robustness outputs:", jev_robustness_paths)
print("Exposure correlation matrix:", exposure_matrix_status)

# Dynamic counterparts to Panels A-D of the long-difference table.
for stale_name in [
    "Robustness_unemployed_panel4.png",
    "Robustness_contracts_panel4.png",
    "Robustness_unemployed_panel5.png",
    "Robustness_contracts_panel5.png",
    "Exposure_correlation_panelA.png",
    "Exposure_correlation_panelB.png",
    "Binary_unemployed_panel1.png",
    "Binary_unemployed_panel2.png",
    "Binary_contracts_panel1.png",
    "Binary_contracts_panel2.png",
]:
    (PAPER_FIGURES_DIR / stale_name).unlink(missing_ok=True)

robust_event_specs = [
    ("preferred_cno1_month", "ln_parados", "ln_contratos"),
    ("log_plus_one_cno1_month", "ln_parados_p1", "ln_contratos_p1"),
    ("cosine_weighted_cno1_month", "ln_parados", "ln_contratos"),
]
for panel, (spec, outcome_u, outcome_c) in enumerate(robust_event_specs, start=1):
    unemployed_ylim = (-0.10, 0.20) if panel == 4 else (-0.05, 0.05)
    unemployed_step = 0.05 if panel == 4 else 0.025
    render_event_file(TABLES_DIR / f"twfe_event_{spec}_{outcome_u}.csv", PAPER_FIGURES_DIR / f"Robustness_unemployed_panel{panel}.png", unemployed_ylim, ylabel="Estimated marginal effect", ytick_step=unemployed_step)
    render_event_file(TABLES_DIR / f"twfe_event_{spec}_{outcome_c}.csv", PAPER_FIGURES_DIR / f"Robustness_contracts_panel{panel}.png", (-0.30, 0.30), ylabel="Estimated marginal effect", ytick_step=0.10)

# Leave-one-CNO1-out sensitivity for the preferred long difference.
leaveout_rows = []
for omitted in range(10):
    for outcome in ["ln_parados", "ln_contratos"]:
        row = longdiff_row(f"leaveout_cno1_{omitted}", outcome)
        if row is None:
            raise FileNotFoundError(
                f"Missing leave-one-family output for CNO1={omitted}, {outcome}."
            )
        leaveout_rows.append({
            "omitted_cno1": omitted,
            "outcome": outcome,
            "estimate": row.estimate,
            "se": row.se,
            "ci_low": row.ci_low,
            "ci_high": row.ci_high,
            "observations": row.observations,
        })
leaveout = pd.DataFrame(leaveout_rows)
leaveout.to_csv(TABLES_DIR / "leave_one_cno1_summary_v1.csv", index=False)
for outcome, filename, ylim in [
    ("ln_parados", "LeaveOneCNO1_unemployed.png", (-0.03, 0.06)),
    ("ln_contratos", "LeaveOneCNO1_contracts.png", (-0.06, 0.06)),
]:
    frame = leaveout.loc[leaveout["outcome"].eq(outcome)].copy()
    fig, ax = plt.subplots(figsize=(5.8, 3.6))
    ax.axhline(0, color=GREY, linewidth=0.7)
    ax.vlines(
        frame["omitted_cno1"], frame["ci_low"], frame["ci_high"],
        color=SKY, linewidth=2.5,
    )
    ax.scatter(frame["omitted_cno1"], frame["estimate"], color=NAVY, s=22)
    ax.set_xticks(range(10))
    ax.set_xlabel("Omitted CNO1 family")
    ax.set_ylabel("Long-difference estimate")
    ax.set_ylim(*ylim)
    fig.tight_layout()
    fig.savefig(PAPER_FIGURES_DIR / filename, bbox_inches="tight")
    plt.close(fig)

# Binary-treatment robustness: exposure above 0.1169 versus all exposure
# at or below 0.1169, retaining middle-exposure occupations as controls.
binary_columns = [
    longdiff_row("binary_high_all_benchmark", "ln_parados"),
    longdiff_row("binary_high_all_preferred", "ln_parados"),
    longdiff_row("binary_high_all_preferred_cluster_cno3", "ln_parados"),
    longdiff_row("binary_high_all_benchmark", "ln_contratos"),
    longdiff_row("binary_high_all_preferred", "ln_contratos"),
    longdiff_row("binary_high_all_preferred_cluster_cno3", "ln_contratos"),
]
if any(row is None for row in binary_columns):
    raise FileNotFoundError("Missing one or more binary-treatment TWFE outputs.")

binary_tex = [
    r"\begin{table}[!htbp]",
    r"\centering",
    r"\caption{Binary-treatment robustness checks}",
    r"\label{tab:v1_binary_treatment}",
    r"\begin{threeparttable}",
    r"\scriptsize",
    r"\setlength{\tabcolsep}{3.5pt}",
    r"\begin{tabular}{lcccccc}",
    r"\toprule",
    r"& \multicolumn{3}{c}{\# of registered unemployed} & \multicolumn{3}{c}{\# of new contracts} \\",
    r"\cmidrule(lr){2-4}\cmidrule(lr){5-7}",
    r"& (1) & (2) & (3) & (4) & (5) & (6) \\",
    r"\midrule",
    "Binary treatment & "
        + " & ".join(coefficient(row.estimate, row.se) for row in binary_columns)
        + r" \\",
    " & " + " & ".join(standard_error(row.se) for row in binary_columns) + r" \\",
    r"\midrule",
    "Impact of treatment (percent) & " + " & ".join(f"{100*row.estimate:.1f}" for row in binary_columns) + r" \\",
    r"CNO4 FE & Yes & Yes & Yes & Yes & Yes & Yes \\",
    r"Year-month FE & Yes & No & No & Yes & No & No \\",
    r"CNO1 $\times$ year-month FE & No & Yes & Yes & No & Yes & Yes \\",
    r"Clustered standard errors & CNO4 & CNO4 & CNO3 & CNO4 & CNO4 & CNO3 \\",
    "Observations & "
        + " & ".join(f"{int(row.observations):,}" for row in binary_columns)
        + r" \\",
    r"\bottomrule",
    r"\end{tabular}",
    r"\begin{tablenotes}[flushleft]",
    r"\footnotesize",
    r"\item \emph{Notes:} Entries are occupation-level long-difference estimates between November 2022 and November 2025. The treatment group consists of occupations with exposure above 0.1169, while all occupations at or below 0.1169 form the control group. Columns 1 and 4 are unconditional first-difference regressions; columns 2 and 5 absorb CNO1 fixed effects; columns 3 and 6 use the same specification with CNO3-clustered standard errors. The impact row reports 100 times the binary-treatment coefficient and carries no significance symbols. $^{***}p<0.01$, $^{**}p<0.05$, and $^{*}p<0.10$.",
    r"\end{tablenotes}",
    r"\end{threeparttable}",
    r"\end{table}",
]
save_tex("binary_treatment_v1.tex", binary_tex)

binary_pretrend_sources = {
    "CNO4": [
        TABLES_DIR / "twfe_binary_pretrend_binary_high_all_preferred_ln_parados.csv",
        TABLES_DIR / "twfe_binary_pretrend_binary_high_all_preferred_ln_contratos.csv",
    ],
    "CNO3": [
        TABLES_DIR / "twfe_binary_pretrend_binary_high_all_preferred_cluster_cno3_ln_parados.csv",
        TABLES_DIR / "twfe_binary_pretrend_binary_high_all_preferred_cluster_cno3_ln_contratos.csv",
    ],
}
binary_pretrend_frames = []
for cluster_level, paths in binary_pretrend_sources.items():
    for path in paths:
        frame = pd.read_csv(require(path))
        frame["cluster_level"] = cluster_level
        binary_pretrend_frames.append(frame)
binary_pretrend = pd.concat(binary_pretrend_frames, ignore_index=True)
binary_pretrend.to_csv(TABLES_DIR / "binary_pretrend_diagnostics_v1.csv", index=False)

def binary_pretrend_pvalue(cluster_level, outcome, window, test):
    match = binary_pretrend.loc[
        binary_pretrend["cluster_level"].eq(cluster_level)
        & binary_pretrend["outcome"].eq(outcome)
        & binary_pretrend["window"].eq(window)
        & binary_pretrend["test"].eq(test),
        "p_value",
    ]
    if len(match) != 1:
        raise ValueError(f"Expected one binary pre-trend result, found {len(match)}")
    return float(match.iloc[0])

binary_pre_lines = [
    r"\begin{table}[H]", r"\centering",
    r"\caption{Pre-treatment diagnostics for the binary-treatment design}",
    r"\label{tab:v1_binary_pretrends}", r"\begin{threeparttable}",
    r"\scriptsize", r"\setlength{\tabcolsep}{4pt}",
    r"\begin{tabular}{llcccc}", r"\toprule",
    r"& & \multicolumn{2}{c}{CNO4 clustering} & \multicolumn{2}{c}{CNO3 clustering} \\",
    r"\cmidrule(lr){3-4}\cmidrule(lr){5-6}",
    r"Outcome & Window & Jointly zero & Equal coefficients & Jointly zero & Equal coefficients \\",
    r"\midrule",
]
for outcome_number, outcome in enumerate(["ln_parados", "ln_contratos"]):
    if outcome_number:
        binary_pre_lines.append(r"\addlinespace")
    for window in ["full_-21_-2", "early_-21_-10", "recent_-10_-2"]:
        values = [
            binary_pretrend_pvalue(cluster, outcome, window, test)
            for cluster in ["CNO4", "CNO3"]
            for test in ["joint_equal_zero", "joint_equal_coefficients"]
        ]
        binary_pre_lines.append(
            f"{labels[outcome]} & {window_labels[window]} & "
            + " & ".join(f"{value:.3f}" for value in values)
            + r" \\"
        )
binary_pre_lines += [
    r"\bottomrule", r"\end{tabular}",
    r"\begin{tablenotes}[flushleft]", r"\footnotesize",
    r"\item \emph{Notes:} Entries are $p$-values from Wald tests of the pre-treatment coefficients in the preferred binary-treatment specification, which includes CNO4 and CNO1-by-year-month fixed effects. Treated occupations have exposure above 0.1169, the 75th percentile of exposure; occupations at or below the cutoff form the comparison group. The joint-nullity test evaluates whether all coefficients in the indicated window equal zero, while the equality test evaluates whether they equal one another. October 2022 is the omitted reference month. Inference allows arbitrary correlation within either CNO4 occupations or CNO3 occupation groups, as indicated.",
    r"\end{tablenotes}", r"\end{threeparttable}", r"\end{table}",
]
save_tex("binary_pretrend_diagnostics_v1.tex", binary_pre_lines)

render_event_file(TABLES_DIR / "twfe_binary_event_binary_high_all_preferred_ln_parados.csv", PAPER_FIGURES_DIR / "Binary_event_unemployed.png", (-0.10, 0.15), ylabel="Estimated treatment effect", ytick_step=0.05)
render_event_file(TABLES_DIR / "twfe_binary_event_binary_high_all_preferred_ln_contratos.csv", PAPER_FIGURES_DIR / "Binary_event_contracts.png", (-0.30, 0.30), ylabel="Estimated treatment effect", ytick_step=0.10)

province_columns = [
    longdiff_row("province_benchmark", "ln_parados"),
    longdiff_row("province_cno1_month", "ln_parados"),
    longdiff_row("province_cno1_month_cluster_cno3", "ln_parados"),
    longdiff_row("province_benchmark", "ln_contratos"),
    longdiff_row("province_cno1_month", "ln_contratos"),
    longdiff_row("province_cno1_month_cluster_cno3", "ln_contratos"),
]
if any(row is None for row in province_columns):
    raise FileNotFoundError("Missing province long-difference outputs.")
province_tex = [
    r"\begin{table}[!htbp]", r"\centering",
    r"\caption{Province-panel robustness}", r"\label{tab:v1_province}",
    r"\begin{threeparttable}", r"\scriptsize", r"\setlength{\tabcolsep}{3.5pt}",
    r"\begin{tabular}{lcccccc}", r"\toprule",
    r"& \multicolumn{3}{c}{\# of registered unemployed} & \multicolumn{3}{c}{\# of new contracts} \\",
    r"\cmidrule(lr){2-4}\cmidrule(lr){5-7}", r"& (1) & (2) & (3) & (4) & (5) & (6) \\", r"\midrule",
    "AI exposure & " + " & ".join(coefficient(row.estimate, row.se) for row in province_columns) + r" \\",
    " & " + " & ".join(standard_error(row.se) for row in province_columns) + r" \\", r"\midrule",
    "Impact of a 10 pp increase (percent) & " + " & ".join(f"{100*row.estimate:.1f}" for row in province_columns) + r" \\",
    r"Province fixed effects & Yes & Yes & Yes & Yes & Yes & Yes \\",
    r"CNO1 fixed effects & No & Yes & Yes & No & Yes & Yes \\",
    r"Clustered standard errors & CNO4 & CNO4 & CNO3 & CNO4 & CNO4 & CNO3 \\",
    "Observations & " + " & ".join(f"{int(row.observations):,}" for row in province_columns) + r" \\",
    r"\bottomrule", r"\end{tabular}", r"\begin{tablenotes}[flushleft]", r"\footnotesize",
    r"\item \emph{Notes:} Entries are province-by-CNO4 long-difference estimates between November 2022 and November 2025. Province fixed effects in the differenced equation are equivalent to province-by-year-month effects in the two-period panel. Columns 2, 3, 5, and 6 additionally absorb CNO1 fixed effects. Standard errors are clustered as indicated. Exposure is measured in 10 percentage-point units, and the impact row reports 100 times the coefficient without additional significance symbols. $^{***}p<0.01$, $^{**}p<0.05$, and $^{*}p<0.10$.",
    r"\end{tablenotes}", r"\end{threeparttable}", r"\end{table}",
]
save_tex("province_robustness_v1.tex", province_tex)
render_event_file(TABLES_DIR / "twfe_event_province_cno1_month_ln_parados.csv", PAPER_FIGURES_DIR / "Province_event_unemployed.png", (-0.05, 0.05), ylabel="Estimated marginal effect", ytick_step=0.025)
render_event_file(TABLES_DIR / "twfe_event_province_cno1_month_ln_contratos.csv", PAPER_FIGURES_DIR / "Province_event_contracts.png", (-0.20, 0.20), ylabel="Estimated marginal effect", ytick_step=0.05)


## 8. Continuous-DiD Alternatives

Organize the three R estimators: unconditional ContDID, CNO1-stratified ContDID, and the two-step control-based CNO1-by-month adjustment. The latter two are closer to the preferred economic comparison, but neither is presented as algebraically equivalent to a covariate-adjusted continuous-DiD estimator.


In [6]:
contdid_specs = [
    ("CNO1-stratified", "cno1_stratified", "ln_parados", "ln_contratos"),
    (
        "Control-based CNO1-month adjustment",
        "control_based_cno1_month",
        "ln_parados_control_adjusted",
        "ln_contratos_control_adjusted",
    ),
    ("Unconditional", "unconditional", "ln_parados", "ln_contratos"),
]

for index, (_, spec, outcome_u, outcome_c) in enumerate(contdid_specs, start=1):
    render_event_file(
        TABLES_DIR / f"contdid_event_{spec}_{outcome_u}.csv",
        PAPER_FIGURES_DIR / f"ContDID_panel{2*index-1}.png",
        (-0.05, 0.15),
    )
    render_event_file(
        TABLES_DIR / f"contdid_event_{spec}_{outcome_c}.csv",
        PAPER_FIGURES_DIR / f"ContDID_panel{2*index}.png",
        (-0.30, 0.30),
        ytick_step=0.10,
    )

contdid_values = []
for label, spec, outcome_u, outcome_c in contdid_specs:
    u = read_first(f"contdid_average_{spec}_{outcome_u}.csv")
    c = read_first(f"contdid_average_{spec}_{outcome_c}.csv")
    if u is None or c is None:
        raise FileNotFoundError(f"Missing ContDID output for {label}.")
    contdid_values.extend([u, c])

cdid_tex = [
    r"\begin{table}[H]",
    r"\centering",
    r"\caption{Continuous-DiD alternatives}",
    r"\label{tab:v1_contdid_alternatives}",
    r"\begin{threeparttable}",
    r"\small",
    r"\setlength{\tabcolsep}{3.5pt}",
    r"\begin{tabular}{lcccccc}",
    r"\toprule",
    r"& \multicolumn{3}{c}{\# of registered unemployed} & \multicolumn{3}{c}{\# of new contracts} \\",
    r"\cmidrule(lr){2-4}\cmidrule(lr){5-7}",
    r"& (1) & (2) & (3) & (4) & (5) & (6) \\",
    r"\midrule",
    "AI exposure & " + " & ".join(
        coefficient(row.estimate, row.std_error)
        for row in [
            contdid_values[0], contdid_values[2], contdid_values[4],
            contdid_values[1], contdid_values[3], contdid_values[5],
        ]
    ) + r" \\",
    " & " + " & ".join(
        standard_error(row.std_error)
        for row in [
            contdid_values[0], contdid_values[2], contdid_values[4],
            contdid_values[1], contdid_values[3], contdid_values[5],
        ]
    ) + r" \\",
    r"\midrule",
    r"CNO1-stratified & Yes & No & No & Yes & No & No \\",
    r"Control-based family-time adjustment & No & Yes & No & No & Yes & No \\",
    r"Unconditional pooled comparison & No & No & Yes & No & No & Yes \\",
    r"\bottomrule",
    r"\end{tabular}",
    r"\begin{tablenotes}[flushleft]",
    r"\footnotesize",
    r"\item \emph{Notes:} Entries average ACRT estimates over event times 1--40 and are scaled to a 10 percentage-point exposure increase. Columns 1 and 4 aggregate family-specific estimates using positive-exposure occupation shares and retain CNO1 families with at least 15 positive- and 5 zero-exposure occupations. Columns 2 and 5 first subtract each supported family's zero-exposure outcome change relative to October 2022. Columns 3 and 6 are unconditional. Standard errors use the dynamic influence-function covariance matrices; for the stratified estimator, disjoint-family covariance matrices are combined with squared aggregation weights. These exercises are alternatives to, not exact reproductions of, the preferred CNO1-by-month TWFE design. $^{***}p<0.01$, $^{**}p<0.05$, and $^{*}p<0.10$.",
    r"\end{tablenotes}",
    r"\end{threeparttable}",
    r"\end{table}",
]
save_tex("contdid_alternatives_v1.tex", cdid_tex)


Built contdid_alternatives_v1.tex


WindowsPath('C:/Users/ngonzalezp/OneDrive - AIREF/Escritorio/Unempolyment_Benefits/ai_unemployment_analysis/code/last_scripts/intermediate/contdid_alternatives_v1.tex')

## 9. Synthetic Difference-in-Differences

Build the preferred CNO1-by-month-adjusted synthetic-DiD outputs. The binary design compares occupations above 0.1169 with a synthetic weighted combination of all occupations at or below 0.1169. The donor audit reports the treated and donor-weighted mean exposure, the exposure contrast, unit weights, and time weights.


In [7]:
def render_sdid_path(source, destination, ylim=None, ytick_step=None):
    frame = pd.read_csv(require(source)).sort_values("event_time")
    counterfactual = frame["counterfactual"]
    fig, ax = plt.subplots(figsize=(6.2, 3.8))
    ax.plot(frame["event_time"], frame["treated"], color=NAVY, linewidth=1.4, label="High exposure")
    ax.plot(
        frame["event_time"], counterfactual,
        color=SKY, linewidth=1.4, label="Synthetic lower-exposure counterfactual",
    )
    ax.axvline(0, color=GREY, linestyle="--", linewidth=0.8)
    ax.set_xlim(EVENT_MIN, EVENT_MAX)
    if ylim is not None:
        ax.set_ylim(*ylim)
        if ytick_step is not None:
            ax.set_yticks(np.arange(ylim[0], ylim[1] + ytick_step / 2, ytick_step))
    ax.set_xlabel("Months relative to November 2022")
    ax.set_ylabel("Log outcome")
    ax.legend(frameon=False, fontsize=8, loc="upper right")
    fig.tight_layout()
    fig.savefig(destination, bbox_inches="tight")
    plt.close(fig)

for spec, prefix in [
    ("expanded_donor_cno1_month", "SDID_adjusted"),
]:
    render_sdid_path(
        TABLES_DIR / f"sdid_paths_{spec}_ln_parados.csv",
        PAPER_FIGURES_DIR / f"{prefix}_path_unemployed.png",
        (6.0, 8.5),
        0.5,
    )
    render_event_file(
        TABLES_DIR / f"sdid_event_{spec}_ln_parados.csv",
        PAPER_FIGURES_DIR / f"{prefix}_event_unemployed.png",
        (-0.05, 0.10),
        ylabel="Synthetic DID effect",
        ytick_step=0.025,
    )
    render_sdid_path(
        TABLES_DIR / f"sdid_paths_{spec}_ln_contratos.csv",
        PAPER_FIGURES_DIR / f"{prefix}_path_contracts.png",
        (3.5, 7.5),
        1.0,
    )
    render_event_file(
        TABLES_DIR / f"sdid_event_{spec}_ln_contratos.csv",
        PAPER_FIGURES_DIR / f"{prefix}_event_contracts.png",
        (-0.25, 0.25),
        ylabel="Synthetic DID effect",
    )

sdid_columns = [
    read_first("sdid_average_expanded_donor_cno1_month_ln_parados.csv"),
    read_first("sdid_average_expanded_donor_cno1_month_ln_contratos.csv"),
]
if any(row is None for row in sdid_columns):
    raise FileNotFoundError("Run the full V1 Stata SDID section before this cell.")

sdid_tex = [
    r"\begin{table}[H]",
    r"\centering",
    r"\caption{Synthetic difference-in-differences estimates}",
    r"\label{tab:v1_sdid}",
    r"\begin{threeparttable}",
    r"\small",
    r"\setlength{\tabcolsep}{4pt}",
    r"\begin{tabular}{lcc}",
    r"\toprule",
    r"& \multicolumn{1}{c}{\# of registered unemployed} & \multicolumn{1}{c}{\# of new contracts} \\",
    r"\cmidrule(lr){2-2}\cmidrule(lr){3-3}",
    r"& (1) & (2) \\",
    r"\midrule",
    "High exposure & " + " & ".join(coefficient(row.estimate, row.se) for row in sdid_columns) + r" \\",
    " & " + " & ".join(standard_error(row.se) for row in sdid_columns) + r" \\",
    "Impact (percent) & " + " & ".join(f"{100*(math.exp(row.estimate)-1):.1f}" for row in sdid_columns) + r" \\",
    r"\midrule",
    r"All lower-exposure occupations eligible as donors & Yes & Yes \\",
    r"CNO1 $\times$ month residualization & Yes & Yes \\",
    "Treated occupations & " + " & ".join(f"{int(row.treated_units):,}" for row in sdid_columns) + r" \\",
    "Donor occupations & " + " & ".join(f"{int(row.donor_units):,}" for row in sdid_columns) + r" \\",
    "Observations & " + " & ".join(f"{int(row.observations):,}" for row in sdid_columns) + r" \\",
    r"\bottomrule",
    r"\end{tabular}",
    r"\begin{tablenotes}[flushleft]",
    r"\footnotesize",
    r"\item \emph{Notes:} Treated occupations have nearest-neighbor exposure above 0.1169; every occupation at or below the cutoff is retained as a potential donor. Thus the estimand contrasts upper-tail exposure with a synthetic lower-exposure mix, not with zero exposure alone. Both columns residualize outcomes on CNO1-by-month indicators before constructing the synthetic comparison. Unit weights minimize pre-treatment discrepancies between treated and donor outcomes, while time weights emphasize pre-treatment months that best predict post-treatment donor outcomes. Standard errors in parentheses use placebo inference with 500 repetitions. $^{***}p<0.01$, $^{**}p<0.05$, and $^{*}p<0.10$.",
    r"\end{tablenotes}",
    r"\end{threeparttable}",
    r"\end{table}",
]
save_tex("sdid_estimates_v1.tex", sdid_tex)

english_titles = pd.read_csv(
    PROCESSED_DIR / "cno4_english_titles.csv", dtype={"cno4": str}
)
english_titles["cno4"] = english_titles["cno4"].str.zfill(4)
occupation_level = (
    total_panel[["cno4", "exposure_nearest"]]
    .drop_duplicates("cno4")
    .merge(english_titles[["cno4", "occupation_title_english"]], on="cno4", how="left")
)
treated = occupation_level.loc[occupation_level["exposure_nearest"] > HIGH_CUTOFF].copy()
treated = treated.sort_values("exposure_nearest", ascending=False)

omega_u = pd.read_csv(
    require(TABLES_DIR / "sdid_omega_expanded_donor_cno1_month_ln_parados.csv"),
    dtype={"cno4": str},
)
omega_c = pd.read_csv(
    require(TABLES_DIR / "sdid_omega_expanded_donor_cno1_month_ln_contratos.csv"),
    dtype={"cno4": str},
)
for frame in [omega_u, omega_c]:
    frame["cno4"] = frame["cno4"].str.zfill(4)
donors = (
    omega_u[["cno4", "exposure_nearest", "occupation_title", "omega"]]
    .rename(columns={"omega": "weight_unemployed"})
    .merge(
        omega_c[["cno4", "omega"]].rename(columns={"omega": "weight_contracts"}),
        on="cno4",
        how="outer",
    )
    .fillna({"weight_unemployed": 0, "weight_contracts": 0})
)
donors = donors.loc[
    (donors["weight_unemployed"].abs() > 1e-10)
    | (donors["weight_contracts"].abs() > 1e-10)
].copy()
donors["max_weight"] = donors[["weight_unemployed", "weight_contracts"]].max(axis=1)
donors = donors.sort_values("max_weight", ascending=False)
donors.to_csv(TABLES_DIR / "sdid_positive_weight_donors_v1.csv", index=False)
treated.to_csv(TABLES_DIR / "sdid_treated_occupations_v1.csv", index=False)

weight_audit = pd.DataFrame([
    {
        "outcome": "ln_parados",
        "weight_sum": omega_u["omega"].sum(),
        "positive_weight_donors": int((omega_u["omega"] > 1e-10).sum()),
        "effective_donors": 1 / np.square(omega_u["omega"]).sum(),
        "largest_weight": omega_u["omega"].max(),
        "top_10_weight_share": omega_u.nlargest(10, "omega")["omega"].sum(),
        "treated_mean_exposure": omega_u["treated_mean_exposure"].iloc[0],
        "donor_weighted_exposure": omega_u["donor_weighted_exposure"].iloc[0],
        "exposure_contrast": omega_u["exposure_contrast"].iloc[0],
    },
    {
        "outcome": "ln_contratos",
        "weight_sum": omega_c["omega"].sum(),
        "positive_weight_donors": int((omega_c["omega"] > 1e-10).sum()),
        "effective_donors": 1 / np.square(omega_c["omega"]).sum(),
        "largest_weight": omega_c["omega"].max(),
        "top_10_weight_share": omega_c.nlargest(10, "omega")["omega"].sum(),
        "treated_mean_exposure": omega_c["treated_mean_exposure"].iloc[0],
        "donor_weighted_exposure": omega_c["donor_weighted_exposure"].iloc[0],
        "exposure_contrast": omega_c["exposure_contrast"].iloc[0],
    },
])
weight_audit.to_csv(TABLES_DIR / "sdid_weight_audit_v1.csv", index=False)
assert np.allclose(weight_audit["weight_sum"], 1, atol=1e-6)

donor_tex = [
    r"\small",
    r"\begin{longtable}{p{0.08\textwidth}>{\raggedright\arraybackslash}p{0.43\textwidth}rrr}",
    r"\caption{Treated occupations and positive-weight synthetic-DiD donors}\label{tab:v1_sdid_donors}\\",
    r"\toprule",
    r"CNO4 & Occupation & Exposure & \shortstack{Weight: \\ unemployed} & \shortstack{Weight: \\ contracts} \\",
    r"\midrule",
    r"\endfirsthead",
    r"\multicolumn{5}{c}{\tablename\ \thetable\ -- continued} \\",
    r"\toprule",
    r"CNO4 & Occupation & Exposure & \shortstack{Weight: \\ unemployed} & \shortstack{Weight: \\ contracts} \\",
    r"\midrule",
    r"\endhead",
    r"\multicolumn{5}{l}{\textit{Panel A. Treated occupations}} \\",
]
for row in treated.itertuples():
    donor_tex.append(
        f"{row.cno4} & {latex_escape(row.occupation_title_english)} & "
        f"{row.exposure_nearest:.3f} & & \\\\"
    )
donor_tex += [
    r"\midrule",
    r"\multicolumn{5}{l}{\textit{Panel B. Donors with positive weight in at least one outcome}} \\",
]
for row in donors.itertuples():
    title = row.occupation_title
    donor_tex.append(
        f"{row.cno4} & {latex_escape(title)} & {row.exposure_nearest:.3f} & "
        f"{row.weight_unemployed:.4f} & {row.weight_contracts:.4f} \\\\"
    )
donor_tex += [
    r"\midrule",
    r"\multicolumn{5}{p{0.93\textwidth}}{\footnotesize \emph{Notes:} Panel A lists occupations above the fixed 0.1169 cutoff. Panel B is ordered by the larger of the two outcome-specific weights and omits donors receiving zero weight in both estimations. Occupation descriptions are English translations of official CNO titles. Donor weights sum to one separately by outcome.} \\",
    r"\bottomrule",
    r"\end{longtable}",
    r"\normalsize",
]
save_tex("sdid_donor_weights_v1.tex", donor_tex)

weight_summary_tex = [
    r"\begin{table}[!htbp]",
    r"\centering",
    r"\caption{Synthetic-DiD donor-weight diagnostics}",
    r"\label{tab:v1_sdid_weight_diagnostics}",
    r"\begin{threeparttable}",
    r"\small",
    r"\begin{tabular}{lcc}",
    r"\toprule",
    r"& \# of registered unemployed & \# of new contracts \\",
    r"\midrule",
]
weight_labels = {
    "positive_weight_donors": "Positive-weight donors",
    "effective_donors": "Effective number of donors",
    "largest_weight": "Largest donor weight",
    "top_10_weight_share": "Share held by top 10 donors",
    "treated_mean_exposure": "Mean treated exposure",
    "donor_weighted_exposure": "Donor-weighted exposure",
    "exposure_contrast": "Exposure contrast",
}
u_audit = weight_audit.loc[weight_audit["outcome"].eq("ln_parados")].iloc[0]
c_audit = weight_audit.loc[weight_audit["outcome"].eq("ln_contratos")].iloc[0]
for variable, label in weight_labels.items():
    if variable == "positive_weight_donors":
        values = [f"{int(u_audit[variable]):,}", f"{int(c_audit[variable]):,}"]
    elif variable == "effective_donors":
        values = [f"{u_audit[variable]:.1f}", f"{c_audit[variable]:.1f}"]
    else:
        values = [f"{u_audit[variable]:.3f}", f"{c_audit[variable]:.3f}"]
    weight_summary_tex.append(f"{label} & {values[0]} & {values[1]} \\\\")
weight_summary_tex += [
    r"\bottomrule",
    r"\end{tabular}",
    r"\begin{tablenotes}[flushleft]",
    r"\footnotesize",
    r"\item \emph{Notes:} The effective number of donors is \(1/\sum_j\widehat{\omega}_j^2\). The exposure contrast is the treated mean minus the donor-weighted mean. Calculations use the CNO1-by-month-adjusted expanded-donor synthetic-DiD specification.",
    r"\end{tablenotes}",
    r"\end{threeparttable}",
    r"\end{table}",
]
save_tex("sdid_weight_diagnostics_v1.tex", weight_summary_tex)

lambda_rows = []
for specification, prefix in [
    ("expanded_donor_cno1_month", "adjusted"),
]:
    for outcome, outcome_label in [
        ("ln_parados", "unemployed"),
        ("ln_contratos", "contracts"),
    ]:
        frame = pd.read_csv(
            require(TABLES_DIR / f"sdid_lambda_{specification}_{outcome}.csv")
        ).sort_values("event_time")
        lambda_rows.append(
            frame.assign(specification_label=prefix, outcome_label=outcome_label)
        )
        fig, ax = plt.subplots(figsize=(5.8, 3.6))
        ax.bar(frame["event_time"], frame["lambda"], color=SKY, width=0.78)
        ax.axhline(0, color=GREY, linewidth=0.7)
        ax.set_xlim(frame["event_time"].min() - 0.8, -0.2)
        ax.set_xlabel("Months relative to November 2022")
        ax.set_ylabel(r"Time weight, $\lambda_t$")
        fig.tight_layout()
        fig.savefig(
            PAPER_FIGURES_DIR / f"SDID_lambda_{prefix}_{outcome_label}.png",
            bbox_inches="tight",
        )
        plt.close(fig)
lambda_frame = pd.concat(lambda_rows, ignore_index=True)
lambda_frame.to_csv(TABLES_DIR / "sdid_time_weights_v1.csv", index=False)
display(weight_audit)


Built sdid_estimates_v1.tex
Built sdid_donor_weights_v1.tex
Built sdid_weight_diagnostics_v1.tex


,outcome,weight_sum,positive_weight_donors,effective_donors,largest_weight,top_10_weight_share,treated_mean_exposure,donor_weighted_exposure,exposure_contrast
0,ln_parados,1.0,377,374.602713,0.004263,0.033821,0.289030,0.017141,0.271888
1,ln_contratos,1.0,363,362.167325,0.003284,0.030824,0.293291,0.016720,0.276571


## 10. HonestDiD, TWFE Heterogeneity, and Output Inventory

Retain the HonestDiD smoothness sensitivity intervals as internal outputs. Summarize age and gender heterogeneity using the same three TWFE specifications as the baseline long-difference table, produce preferred CNO1-by-month event-study panels with common outcome-specific scales, and export subgroup pre-trend diagnostics. Finally, create a machine-readable map of every paper-facing artifact.


In [8]:
for outcome, filename in [
    ("ln_parados", "HonestDID_unemployed.png"),
    ("ln_contratos", "HonestDID_contracts.png"),
]:
    frame = pd.read_csv(
        require(TABLES_DIR / f"honestdid_twfe_benchmark_{outcome}.csv")
    )
    frame = frame.loc[pd.to_numeric(frame["M"], errors="coerce").notna()].copy()
    frame["M"] = pd.to_numeric(frame["M"])
    fig, ax = plt.subplots(figsize=(5.8, 3.6))
    ax.axhline(0, color=GREY, linewidth=0.7)
    ax.vlines(frame["M"], frame["ci_low"], frame["ci_high"], color=SKY, linewidth=3)
    ax.scatter(frame["M"], (frame["ci_low"] + frame["ci_high"]) / 2, color=NAVY, s=20)
    ax.set_xlabel(r"Smoothness restriction, $M$")
    ax.set_ylabel("Robust confidence interval")
    fig.tight_layout()
    fig.savefig(PAPER_FIGURES_DIR / filename, bbox_inches="tight")
    plt.close(fig)

heterogeneity_groups = [
    ("Age: under 30", "age", "lt18_to_29"),
    ("Age: 30--39", "age", "30-39"),
    ("Age: 40 or older", "age", "40_to_gt44"),
    ("Gender: men", "gender", "hombre"),
    ("Gender: women", "gender", "mujer"),
]

def heterogeneity_spec(dimension, tag, specification):
    suffix = {
        "benchmark": "benchmark",
        "preferred": "cno1_month",
        "preferred_cno3": "cno1_month_cluster_cno3",
    }[specification]
    return f"{dimension}_{tag}_{suffix}"

heterogeneity_results = {}
for label, dimension, tag in heterogeneity_groups:
    rows = []
    for outcome in ["ln_parados", "ln_contratos"]:
        for specification in ["benchmark", "preferred", "preferred_cno3"]:
            row = longdiff_row(
                heterogeneity_spec(dimension, tag, specification), outcome
            )
            if row is None:
                raise FileNotFoundError(
                    f"Missing heterogeneity long difference: {label}, {outcome}, {specification}"
                )
            rows.append(row)
    heterogeneity_results[label] = rows

hetero_tex = [
    r"\begin{table}[!htbp]", r"\centering",
    r"\caption{Heterogeneity in the long-run effects of AI exposure}",
    r"\label{tab:v1_heterogeneity}", r"\begin{threeparttable}",
    r"\scriptsize", r"\setlength{\tabcolsep}{3.5pt}",
    r"\begin{tabular}{lcccccc}", r"\toprule",
    r"& \multicolumn{3}{c}{\# of registered unemployed} & \multicolumn{3}{c}{\# of new contracts} \\",
    r"\cmidrule(lr){2-4}\cmidrule(lr){5-7}",
    r"& (1) & (2) & (3) & (4) & (5) & (6) \\", r"\midrule",
]
for group_number, (label, _, _) in enumerate(heterogeneity_groups):
    if group_number:
        hetero_tex.append(r"\addlinespace")
    rows = heterogeneity_results[label]
    hetero_tex.append(rf"\multicolumn{{7}}{{l}}{{\textit{{{label}}}}} \\")
    hetero_tex.append(
        "AI exposure & "
        + " & ".join(coefficient(row.estimate, row.se) for row in rows)
        + r" \\"
    )
    hetero_tex.append(
        " & " + " & ".join(standard_error(row.se) for row in rows) + r" \\"
    )
    hetero_tex.append(
        "Impact of a 10 pp increase (percent) & "
        + " & ".join(f"{100*row.estimate:.1f}" for row in rows)
        + r" \\"
    )
    hetero_tex.append(
        "Observations & "
        + " & ".join(f"{int(row.observations):,}" for row in rows)
        + r" \\"
    )
hetero_tex += [
    r"\midrule",
    r"CNO1 $\times$ year-month FE & No & Yes & Yes & No & Yes & Yes \\",
    r"Clustered standard errors & CNO4 & CNO4 & CNO3 & CNO4 & CNO4 & CNO3 \\",
    r"\bottomrule", r"\end{tabular}",
    r"\begin{tablenotes}[flushleft]", r"\footnotesize",
    r"\item \emph{Notes:} Each panel is estimated on the indicated subgroup and reports an occupation-level long difference between November 2022 and November 2025. Columns 1 and 4 report unconditional first-difference regressions. Columns 2, 3, 5, and 6 compare occupations within CNO1 families by absorbing CNO1 fixed effects in the differenced equation, the two-period analogue of CNO1-by-year-month effects. Exposure is measured in 10 percentage-point units. The impact rows report 100 times the corresponding log-point coefficient and carry no additional significance symbols. Standard errors are clustered as indicated. $^{***}p<0.01$, $^{**}p<0.05$, and $^{*}p<0.10$.",
    r"\end{tablenotes}", r"\end{threeparttable}", r"\end{table}",
]
save_tex("heterogeneity_v1.tex", hetero_tex)

# Pre-treatment diagnostics for the preferred subgroup event studies.
heterogeneity_pretrend_frames = []
for label, dimension, tag in heterogeneity_groups:
    for cluster_level, specification in [
        ("CNO4", "preferred"), ("CNO3", "preferred_cno3")
    ]:
        for outcome in ["ln_parados", "ln_contratos"]:
            path = TABLES_DIR / (
                f"twfe_pretrend_{heterogeneity_spec(dimension, tag, specification)}_{outcome}.csv"
            )
            frame = pd.read_csv(require(path))
            frame["group"] = label
            frame["cluster_level"] = cluster_level
            heterogeneity_pretrend_frames.append(frame)
heterogeneity_pretrend = pd.concat(heterogeneity_pretrend_frames, ignore_index=True)
heterogeneity_pretrend.to_csv(
    TABLES_DIR / "heterogeneity_pretrend_diagnostics_v1.csv", index=False
)

def heterogeneity_pretrend_pvalue(group, cluster_level, outcome, window, test):
    match = heterogeneity_pretrend.loc[
        heterogeneity_pretrend["group"].eq(group)
        & heterogeneity_pretrend["cluster_level"].eq(cluster_level)
        & heterogeneity_pretrend["outcome"].eq(outcome)
        & heterogeneity_pretrend["window"].eq(window)
        & heterogeneity_pretrend["test"].eq(test),
        "p_value",
    ]
    if len(match) != 1:
        raise ValueError(f"Expected one subgroup pre-trend result, found {len(match)}")
    return float(match.iloc[0])

hetero_pre_tex = [
    r"\begingroup", r"\scriptsize", r"\setlength{\tabcolsep}{3.5pt}",
    r"\begin{longtable}{llcccc}",
    r"\caption{Pre-treatment diagnostics for the heterogeneity analysis}\label{tab:v1_heterogeneity_pretrends}\\",
    r"\toprule",
    r"& & \multicolumn{2}{c}{CNO4 clustering} & \multicolumn{2}{c}{CNO3 clustering} \\",
    r"\cmidrule(lr){3-4}\cmidrule(lr){5-6}",
    r"Outcome & Window & Jointly zero & Equal coefficients & Jointly zero & Equal coefficients \\",
    r"\midrule", r"\endfirsthead",
    r"\multicolumn{6}{c}{\tablename\ \thetable{} -- continued from previous page} \\",
    r"\toprule",
    r"Outcome & Window & Jointly zero & Equal coefficients & Jointly zero & Equal coefficients \\",
    r"\midrule", r"\endhead",
]
for group_number, (label, _, _) in enumerate(heterogeneity_groups):
    if group_number:
        hetero_pre_tex.append(r"\addlinespace")
    hetero_pre_tex.append(rf"\multicolumn{{6}}{{l}}{{\textit{{{label}}}}} \\")
    for outcome in ["ln_parados", "ln_contratos"]:
        for window in ["full_-21_-2", "early_-21_-10", "recent_-10_-2"]:
            values = [
                heterogeneity_pretrend_pvalue(label, cluster, outcome, window, test)
                for cluster in ["CNO4", "CNO3"]
                for test in ["joint_equal_zero", "joint_equal_coefficients"]
            ]
            hetero_pre_tex.append(
                f"{labels[outcome]} & {window_labels[window]} & "
                + " & ".join(f"{value:.3f}" for value in values)
                + r" \\"
            )
hetero_pre_tex += [
    r"\bottomrule", r"\end{longtable}",
    r"\begin{minipage}{0.94\textwidth}\footnotesize\emph{Notes:} Entries are $p$-values from Wald tests of the pre-treatment coefficients in subgroup-specific continuous-treatment regressions with CNO4 and CNO1-by-year-month fixed effects. The joint-nullity test evaluates whether all coefficients in the indicated window equal zero; the equality test evaluates whether they equal one another. October 2022 is the omitted reference month. Inference allows arbitrary correlation within CNO4 occupations or CNO3 occupation groups, as indicated.\end{minipage}",
    r"\endgroup",
]
save_tex("heterogeneity_pretrend_diagnostics_v1.tex", hetero_pre_tex)

# Preferred subgroup event studies. All panels for a given outcome share a
# common vertical range to preserve visual comparability.
for stale_path in PAPER_FIGURES_DIR.glob("Heterogeneity_*_contdid_panel*.png"):
    stale_path.unlink(missing_ok=True)
for stale_path in PAPER_FIGURES_DIR.glob("Heterogeneity_*_sdid_panel*.png"):
    stale_path.unlink(missing_ok=True)
for index, (_, dimension, tag) in enumerate(heterogeneity_groups, start=1):
    specification = heterogeneity_spec(dimension, tag, "preferred")
    render_event_file(
        TABLES_DIR / f"twfe_event_{specification}_ln_parados.csv",
        PAPER_FIGURES_DIR / f"Heterogeneity_unemployed_panel{index}.png",
        (-0.05, 0.075), ylabel="Estimate", ytick_step=0.025,
    )
    render_event_file(
        TABLES_DIR / f"twfe_event_{specification}_ln_contratos.csv",
        PAPER_FIGURES_DIR / f"Heterogeneity_contracts_panel{index}.png",
        (-0.30, 0.30), ylabel="Estimate", ytick_step=0.10,
    )

artifact_rows = []
for path in sorted(PAPER_FIGURES_DIR.glob("*.png")):
    artifact_rows.append({"type": "figure", "name": path.name, "path": str(path)})
for path in sorted(TABLES_DIR.glob("*.tex")):
    artifact_rows.append({"type": "table", "name": path.name, "path": str(path)})
artifact_index = pd.DataFrame(artifact_rows)
artifact_index.to_csv(TABLES_DIR / "paper_artifact_index_v1.csv", index=False)
display(artifact_index)


Built heterogeneity_v1.tex


Built heterogeneity_pretrend_diagnostics_v1.tex


,type,name,path
0,figure,Binary_event_contracts.png,C:\Users\ngonzalezp\OneDrive - AIREF\Escritori...
1,figure,Binary_event_unemployed.png,C:\Users\ngonzalezp\OneDrive - AIREF\Escritori...
2,figure,ContDID_panel1.png,C:\Users\ngonzalezp\OneDrive - AIREF\Escritori...
3,figure,ContDID_panel2.png,C:\Users\ngonzalezp\OneDrive - AIREF\Escritori...
4,figure,ContDID_panel3.png,C:\Users\ngonzalezp\OneDrive - AIREF\Escritori...
...,...,...,...
67,table,sdid_donor_weights_v1.tex,C:\Users\ngonzalezp\OneDrive - AIREF\Escritori...
68,table,sdid_estimates_v1.tex,C:\Users\ngonzalezp\OneDrive - AIREF\Escritori...
69,table,sdid_weight_diagnostics_v1.tex,C:\Users\ngonzalezp\OneDrive - AIREF\Escritori...
70,table,table2_main_effects_v1.tex,C:\Users\ngonzalezp\OneDrive - AIREF\Escritori...


## 11. Phase-Based Tables and Final Robustness Outputs

The paper tables pool the post-treatment path into an adjustment period (event times 0--24) and a later period (event times 25--40). All pre-treatment months form the omitted category in these table regressions. This differs from the unrestricted event studies, which normalize monthly coefficients to October 2022. The cell rebuilds the baseline, robustness, province, heterogeneity, and continuous-DiD tables; it also formats the pooled age tests and trailing 12-month contract check.


In [ ]:
from lib.report_outputs import build_phase_outputs

phase_summary = build_phase_outputs(PROJECT_ROOT, estimates_dir=INTERMEDIATE_DIR, output_dir=FINAL_DIR)
display(phase_summary)

# Organize the feminization estimates produced by Stata into the
# paper-facing tables and appendix figures.
from lib.feminization import build_feminization_outputs
feminization_summary = build_feminization_outputs(PROJECT_ROOT, input_dir=INPUT_DIR, estimates_dir=INTERMEDIATE_DIR, output_dir=FINAL_DIR)
display(feminization_summary)


## 12. Production Validation

These checks guard the central empirical contract: a balanced 502-occupation total panel, aligned event windows, a fixed 0.1169 synthetic-DiD cutoff with no omitted middle group, valid donor weights, and complete paper-facing artifacts.


In [10]:
validation = []

def record(check, passed, detail):
    validation.append({"check": check, "passed": bool(passed), "detail": str(detail)})

age_audit = pd.read_csv(INTERMEDIATE_DIR / "sepe_may2024_age_backcast_cell_audit_v1.csv")
province_audit = pd.read_csv(INTERMEDIATE_DIR / "sepe_may2024_province_backcast_cell_audit_v1.csv")
cutoff_audit = pd.read_csv(INTERMEDIATE_DIR / "treatment_cutoff_audit_v1.csv")

record("Age reconstruction audit is complete", len(age_audit) == 502 * 6 * 2, len(age_audit))
record("Province reconstruction audit is complete", len(province_audit) == 502 * 52 * 2, len(province_audit))
record("Total panel has 502 CNO4 units", total_panel["cno4"].nunique() == 502, total_panel["cno4"].nunique())
record("Total panel has 63 months", total_panel["period"].nunique() == 63, total_panel["period"].nunique())

feminization_panel = pd.read_csv(INPUT_DIR / "est_feminization_cno4.csv", dtype={"cno4": str})
record("Feminization panel has one row per CNO4-month", len(feminization_panel) == 502 * 63, len(feminization_panel))
record("EPA feminization index maps all CNO4 occupations", feminization_panel["feminization_2017_2019"].notna().all(), feminization_panel["cno4"].nunique())
record("Median split partitions the feminization panel", set(feminization_panel["feminization_above_median"].dropna().unique()) == {0, 1}, feminization_panel["feminization_above_median"].value_counts().to_dict())
record("Expanded SDID donor pool partitions all occupations", cutoff_audit.loc[0, "treated_above_0_1169"] + cutoff_audit.loc[0, "donors_at_or_below_0_1169"] == 502, cutoff_audit.iloc[0].to_dict())

for estimator, source in [
    ("Preferred TWFE unemployed", INTERMEDIATE_DIR / "twfe_event_preferred_cno1_month_ln_parados.csv"),
    ("Preferred TWFE contracts", INTERMEDIATE_DIR / "twfe_event_preferred_cno1_month_ln_contratos.csv"),
    ("Stratified ContDID unemployed", INTERMEDIATE_DIR / "contdid_event_cno1_stratified_ln_parados.csv"),
    ("Adjusted SDID unemployed", INTERMEDIATE_DIR / "sdid_event_expanded_donor_cno1_month_ln_parados.csv"),
    ("Adjusted SDID contracts", INTERMEDIATE_DIR / "sdid_event_expanded_donor_cno1_month_ln_contratos.csv"),
]:
    frame = pd.read_csv(require(source))
    record(f"{estimator} event window", frame["event_time"].min() == EVENT_MIN and frame["event_time"].max() == EVENT_MAX, (frame["event_time"].min(), frame["event_time"].max()))

if "weight_audit" in globals():
    record("SDID donor weights sum to one", np.allclose(weight_audit["weight_sum"], 1, atol=1e-6), weight_audit[["outcome", "weight_sum"]].to_dict("records"))

for outcome in ["ln_parados", "ln_contratos"]:
    pooled_age = pd.read_csv(require(INTERMEDIATE_DIR / f"age_pooled_phase_{outcome}.csv"))
    record(f"Pooled age tests cover all groups: {outcome}", set(pooled_age["age_group"]) == {"Under 30", "30--39", "40 or older"}, sorted(pooled_age["age_group"].unique()))

validation_frame = pd.DataFrame(validation)
validation_frame.to_csv(INTERMEDIATE_DIR / "production_validation_v1.csv", index=False)
display(validation_frame)
if not validation_frame["passed"].all():
    raise AssertionError("One or more production checks failed.")


,check,passed,detail
0,Age reconstruction audit is complete,True,6024
1,Province reconstruction audit is complete,True,52208
2,Total panel has 502 CNO4 units,True,502
3,Total panel has 63 months,True,63
4,Feminization panel has one row per CNO4-month,True,31626
5,EPA feminization index maps all CNO4 occupations,True,502
6,Median split partitions the feminization panel,True,"{0: 15813, 1: 15813}"
7,Expanded SDID donor pool partitions all occupa...,True,"{'occupations': 502.0, 'zero_exposure_occupati..."
8,Preferred TWFE unemployed event window,True,"(np.int64(-21), np.int64(40))"
9,Preferred TWFE contracts event window,True,"(np.int64(-21), np.int64(40))"


## Final publication directory and TeX dependency audit\n\nCopy only publication-ready LaTeX fragments from intermediate to the flat final directory. Then verify that every figure and table reference in both TeX documents resolves before compilation.


In [ ]:

for tex_path in INTERMEDIATE_DIR.glob("*.tex"):
    shutil.copy2(tex_path, FINAL_DIR / tex_path.name)

import re
main_tex = PROJECT_ROOT / "estimates_results_report_v1.tex"
descriptive_tex = PROJECT_ROOT / "descriptive.tex"

def referenced_artifacts(tex_path):
    text = tex_path.read_text(encoding="utf-8")
    graphics = re.findall(r"\\includegraphics(?:\[[^\]]*\])?\{\\figdir/([^}]+)\}", text)
    inputs = re.findall(r"\\input\{figuresNtables/([^}]+)\}", text)
    return graphics + inputs

checks = []
for tex_path in [main_tex, descriptive_tex]:
    for name in referenced_artifacts(tex_path):
        candidate = FINAL_DIR / name
        if candidate.suffix == "":
            candidate = candidate.with_suffix(".tex")
        checks.append({"document": tex_path.name, "artifact": name, "exists": candidate.exists()})

dependency_manifest = pd.DataFrame(checks)
dependency_manifest.to_csv(INTERMEDIATE_DIR / "tex_dependency_manifest_v1.csv", index=False)
missing = dependency_manifest.loc[~dependency_manifest["exists"]]
if len(missing):
    raise FileNotFoundError("Missing TeX dependencies:\n" + missing.to_string(index=False))

# Keep the publication directory limited to artifacts used by the two TeX
# documents. Retained diagnostics remain available in intermediate/.
referenced = set(dependency_manifest["artifact"])
for path in FINAL_DIR.iterdir():
    if path.is_file() and path.suffix.lower() in {".png", ".tex"} and path.name not in referenced:
        shutil.move(str(path), str(INTERMEDIATE_DIR / path.name))

final_files = sorted(path for path in FINAL_DIR.iterdir() if path.is_file())
pd.DataFrame([
    {"file": path.name, "extension": path.suffix.lower(), "bytes": path.stat().st_size}
    for path in final_files
]).to_csv(INTERMEDIATE_DIR / "final_artifact_manifest_v1.csv", index=False)

print(f"Output tuning complete: {len(final_files)} flat publication artifacts.")
print("Both TeX dependency audits passed. The documents are ready to compile.")
